# Test simGeneOverExpr
Python version of `R/Tin/test_simGeneOverExpr.R`: over expression of one gene with ccFBA (`simGeneOverExpr`): the gene
must use at least a given fraction `pct` of the budget C. Model iAF1260, kcat values of the file `R/Tin/kcat2`
completed with the kcat data set of the R package, C = 0.27, growth on glucose.

The model and the data sets `kcat` and `mw` are read from the `.RData` files of the R package (as `data(...)` in R), so the
reaction ids are as in R (e.g. `R_EX_glc_e_`). In R the bounds are changed in the irreversible model `mod2`; in python
`cfba_moment` and `ccFBA_mr` use the bounds of the cobra model, so the bounds are changed in `model` and `mod2` is built
after that (for `getccFBA_sol`).

In [1]:
import os
import pandas as pd

from sybil2cobra import read_RData
from getccFBA_mat import mod2irrev
from simGeneOverExpr import simGeneOverExpr
from cfba_moment import cfba_moment
from ccFBA_mr import ccFBA_mr
from getccFBA_sol import getccFBA_sol

pkg = os.path.join('R', 'sybilccFBA')
kcat = read_RData(os.path.join(pkg, 'data', 'kcat.RData'))['kcat']
mw = read_RData(os.path.join(pkg, 'data', 'mw.RData'))['mw']
model = read_RData(os.path.join(pkg, 'data', 'iAF1260.RData'))['iAF1260']
print(model.id, len(model.reactions), 'reactions', len(model.genes), 'genes')

Ec_iAF1260 2382 reactions 1261 genes


### Read the new kcat file
`kcat2` (rxn_id, val, dirxn) is completed with the rows of `kcat` whose reaction and direction are not in `kcat2`.

In [2]:
kcat2_tmp = pd.read_csv(os.path.join('R', 'Tin', 'kcat2'), sep=r'\s+', header=None, names=['rxn_id', 'val', 'dirxn'])
kcat = kcat.astype({'rxn_id': str, 'dirxn': int})
missing = kcat[['rxn_id', 'dirxn']].merge(kcat2_tmp[['rxn_id', 'dirxn']], how='left', indicator=True)['_merge'] == 'left_only'
kcat2_tmp = pd.concat([kcat2_tmp, kcat.loc[missing.values, ['rxn_id', 'val', 'dirxn']]], ignore_index=True)
print('kcat2: %d rows, added from kcat: %d, total: %d' % (len(kcat2_tmp) - missing.sum(), missing.sum(), len(kcat2_tmp)))

kcat2: 1529 rows, added from kcat: 155, total: 1684


### Bounds
Glucose uptake open, no glycerol and acetate uptake, oxygen uptake open, ATPM lower bound 0 (in R:
`uppbnd(mod2)` of the `_b` reactions and `lowbnd(mod2)` of `R_ATPM`).

In [3]:
model.reactions.get_by_id('R_EX_glc_e_').lower_bound = -1000
model.reactions.get_by_id('R_EX_glyc_e_').lower_bound = 0
model.reactions.get_by_id('R_EX_ac_e_').lower_bound = 0
model.reactions.get_by_id('R_EX_o2_e_').lower_bound = -1000
model.reactions.get_by_id('R_ATPM').lower_bound = 0
mod2 = mod2irrev(model)
bm_rxn = [r.id for r in model.reactions if r.objective_coefficient != 0][0]
print(mod2, bm_rxn)

<IrrevModel Ec_iAF1260: 2957 reactions (575 split), 1668 metabolites> R_Ec_biomass_iAF1260_core_59p81M


## 1. Over expression of b2323 (fabB) with `cfba_moment` (MOMENT)
`absMtfFlx`, `absOExFlx`, `absWTFlx`: sum of absolute fluxes of the minimum total flux solution, of the over
expression solution and of the wild type. `Bm`: growth rate after over expression. `gxMWWT`, `gxMWOEx`: MC (g/gDW) of
the gene in the wild type and after over expression, `MCpctOExUsed`: fraction of C used by the gene.

In [4]:
C_mu = 0.27
gx = 'b2323'
v_Fun = cfba_moment
mw_gx = mw.loc[mw.iloc[:, 0] == gx].iloc[0, 1]
abs_sum = lambda flx: sum(abs(v) for v in flx.values())

allRes = []
for pct in [0.05, 0.1, 0.15, 0.25, 0.5]:  # different percentages
    print(pct)
    RR = simGeneOverExpr(gx, pct, model, mod2, kcat2_tmp, MW=mw, verboseMode=1, RHS=C_mu, medval=3600*22.6, FUN=v_Fun)
    gxMWOEx = RR['OExSol']['geneConc'][gx] * mw_gx
    gxMWWT = RR['WTSol']['geneConc'][gx] * mw_gx
    allRes.append(dict(pct=pct, absMtfFlx=abs_sum(RR['Sol']['rxnFlx']), absOExFlx=abs_sum(RR['OExSol']['rxnFlx']),
                       absWTFlx=abs_sum(RR['WTSol']['rxnFlx']), OverExprEff=RR['OverExprEff'], Bm=RR['objVal'],
                       gxMWWT=gxMWWT, gxMWOEx=gxMWOEx, MCpctOExUsed=gxMWOEx / C_mu))
allRes = pd.DataFrame(allRes)

0.05
Preparing LP ...


Adding variables...
Adding constraints...


Solving LP ...


simGeneOverExpr: MC of gene b2323 =0.029969213 , 11.100 % of C by default
simGeneOverExpr: calculate min total flux
simGeneOverExpr: min total flux #Rxns:2382, to be affected=463 


0.1
Preparing LP ...


Adding variables...
Adding constraints...


... (19 outputs left out)

simGeneOverExpr: MC of gene b2323 =0.029969212 , 11.100 % of C by default
Preparing LP ...


Adding variables...
Adding constraints...


Solving LP ...


simGeneOverExpr: MC of gene b2323 =0.135000 , 50.000 % of C after over expression
simGeneOverExpr: calculate min total flux
simGeneOverExpr: min total flux #Rxns:2382, to be affected=463 


In [5]:
allRes

,pct,absMtfFlx,absOExFlx,absWTFlx,OverExprEff,Bm,gxMWWT,gxMWOEx,MCpctOExUsed
0,0.05,742.340221,742.340221,742.340221,False,0.869362,0.029969,0.029969,0.110997
1,0.10,742.340228,742.340228,742.340228,False,0.869362,0.029969,0.029969,0.110997
2,0.15,709.771792,709.771792,742.340228,True,0.831221,0.029969,0.040500,0.150000
3,0.25,626.269228,626.269228,742.340228,True,0.733430,0.029969,0.067500,0.250000
4,0.50,417.512819,417.512819,742.340228,True,0.488953,0.029969,0.135000,0.500000


b2323 uses about 11% of C in the wild type, so 5% and 10% have no effect. With over expression the rest of the
budget is shared by the other enzymes, and the growth rate decreases in proportion to the budget left for them:
`Bm = Bm_WT * (C - gxMWOEx) / (C - gxMWWT)`.

In [6]:
Bm_WT = allRes['Bm'].iloc[0]
allRes['Bm_expected'] = Bm_WT * (C_mu - allRes['gxMWOEx']) / (C_mu - allRes['gxMWWT'])
allRes[['pct', 'OverExprEff', 'Bm', 'Bm_expected']]

,pct,OverExprEff,Bm,Bm_expected
0,0.05,False,0.869362,0.869362
1,0.10,False,0.869362,0.869362
2,0.15,True,0.831221,0.831221
3,0.25,True,0.733430,0.733430
4,0.50,True,0.488953,0.488953


### Genes' molecular crowding (MC) and percentage of C in descending order
Wild type solution (of the last call).

In [7]:
gene_conc = pd.Series(RR['WTSol']['geneConc'])
Res = mw.set_index(mw.columns[0]).reindex(gene_conc.index)
Res['MCv'] = gene_conc * Res.iloc[:, 0]
Res['mcvpct'] = Res['MCv'] / C_mu
Res.sort_values('MCv', ascending=False).head()

,mw,MCv,mcvpct
b2472,41.268720,0.066616,0.246725
b3774,54.068620,0.031609,0.117070
b2323,42.612995,0.029969,0.110997
b2818,49.194750,0.015973,0.059160
b0241,38.921750,0.009006,0.033356


## 2. Over expression of b0929 (ompF) with the three ccFBA functions
`pct` = 1e-4. The kcat values get the source `'Tin'`. R functions and python versions: `cfba_moment_mr` → `ccFBA_mr`,
`cfba_moment` → `cfba_moment`, `getccFBA_sol` → `getccFBA_sol`.

b0929 is one of the isoenzymes of the outer membrane porins (`b0241 or b0929 or b1377 or b2215`). `getccFBA_sol` uses
only the cheapest isoenzyme of each reaction (b0241), so b0929 has no concentration variable: the over expressed protein
only takes its part of the budget (C is reduced by `pct`). In R this case stops with an error.

In [8]:
kcat2_tmp['src'] = 'Tin'
C_mu = 0.27
gx = 'b0929'; pct = 1e-04
mw_gx = mw.loc[mw.iloc[:, 0] == gx].iloc[0, 1]
res2 = []
for v_Fun in [ccFBA_mr, cfba_moment, getccFBA_sol]:
    print(v_Fun.__name__, pct)
    RR = simGeneOverExpr(gx, pct, model, mod2, kcat2_tmp, MW=mw, verboseMode=1, RHS=C_mu, medval=3600*22.6, FUN=v_Fun)
    WTFlx = RR['WTSol']['rxnFlx']
    res2.append(dict(pct=pct, v_Fun=v_Fun.__name__, absMtfFlx=abs_sum(RR['Sol']['rxnFlx']),
                     absOExFlx=abs_sum(RR['OExSol']['rxnFlx']), absWTFlx=abs_sum(WTFlx),
                     OverExprEff=RR['OverExprEff'], Bmwt=WTFlx[bm_rxn], Bm=RR['objVal'],
                     gxMWWT=RR['gxMC_WT'], gxMWOEx=RR['gxMC_OEx'], MCpctOExUsed=RR['gxMC_OEx'] / C_mu))
    print(pd.DataFrame(res2[-1:]).to_string())
res2 = pd.DataFrame(res2)

ccFBA_mr 0.0001
Preparing LP ...


Adding variables...
Adding constraints...


Solving LP ...


simGeneOverExpr: MC of gene b0929 =0.000000000 , 0.000 % of C by default
Preparing LP ...


Adding variables...
Adding constraints...


Solving LP ...


... (12 outputs left out)

Preparing LP ... 


LP: 5277 rows, 6028 columns (1064 genes, 1782 multi-functional parts, 225 complexes)
Solving LP ...


optimal
simGeneOverExpr: MC of gene b0929 =0.000027 , 0.010 % of C after over expression
simGeneOverExpr: calculate min total flux
simGeneOverExpr: min total flux #Rxns:2382, to be affected=463 


      pct         v_Fun   absMtfFlx   absOExFlx    absWTFlx  OverExprEff    Bmwt        Bm  gxMWWT   gxMWOEx  MCpctOExUsed
0  0.0001  getccFBA_sol  532.040045  532.040045  532.093254         True  0.6236  0.623538     0.0  0.000027        0.0001


In [9]:
res2['Bm_change_pct'] = 100 * (res2['Bm'] - res2['Bmwt']) / res2['Bmwt']
res2

,pct,v_Fun,absMtfFlx,absOExFlx,absWTFlx,OverExprEff,Bmwt,Bm,gxMWWT,gxMWOEx,MCpctOExUsed,Bm_change_pct
0,0.0001,ccFBA_mr,604.781559,604.781559,604.782192,True,0.634723,0.634722,0.0,0.000027,0.0001,-0.000105
1,0.0001,cfba_moment,742.339452,742.339452,742.340228,True,0.869362,0.869361,0.0,0.000027,0.0001,-0.000105
2,0.0001,getccFBA_sol,532.040045,532.040045,532.093254,True,0.623600,0.623538,0.0,0.000027,0.0001,-0.010000


With `cfba_moment` and `ccFBA_mr` (`cfba_moment_mr`) the isoenzymes are summed, so the over expressed ompF replaces other
porins and the growth rate hardly changes. With `getccFBA_sol` the protein is not used and the growth rate decreases by
the fraction of the budget it takes (0.01%).